In [24]:
import pymorphy3
import re
from collections import Counter

morph = pymorphy3.MorphAnalyzer()

In [25]:
with open('dom.txt', 'r', encoding='cp1251') as file:
    corpus = file.readlines()

In [52]:
def extract_clean_text(line):
    return ' '.join([word.split('>')[1].split('<')[0] for word in line.split() if '>' in word and '<' in word])

In [53]:
def format_results(results):
    return '\n'.join(results)

In [54]:
def count_token_frequency(token):
    counter = 0
    for line in corpus:
        if token in line:
            counter += 1
    return counter

In [55]:
def search_by_token(token, max_results=5):
    results = []
    for line in corpus:
        if token in line:
            results.append(extract_clean_text(line))
            if len(results) >= max_results:
                break
    frequency = count_token_frequency(token)
    return f"Frequency: {frequency}\n{format_results(results)}"

In [56]:
def count_lemma_frequency(lemma):
    counter = 0
    for line in corpus:
        words = line.split()
        for word in words:
            clean_word = word.split('>')[1].split('<')[0] if '>' in word and '<' in word else word
            parsed = morph.parse(clean_word.strip('.,!?\"\'\"'))[0]
            if parsed.normal_form == lemma:
                counter += 1
                break
    return counter

In [57]:
def search_by_lemma(lemma, max_results=5):
    results = []
    for line in corpus:
        words = line.split()
        for word in words:
            clean_word = word.split('>')[1].split('<')[0] if '>' in word and '<' in word else word
            parsed = morph.parse(clean_word.strip('.,!?\"\'\"'))[0]
            if parsed.normal_form == lemma:
                results.append(extract_clean_text(line))
                break
        if len(results) >= max_results:
            break
    frequency = count_lemma_frequency(lemma)
    return f"Frequency: {frequency}\n{format_results(results)}"

In [58]:
def count_tag_frequency(tag):
    counter = 0
    for line in corpus:
        if tag in line:
            counter += 1
    return counter

In [59]:
def search_by_semantic_tag(tag, max_results=5):
    results = []
    for line in corpus:
        for word in line.split():
            if re.search(f"sem=['\"][^'\"]*{tag}[^'\"]*['\"]", word):
                clean = word.split('>')[1].split('<')[0]
                results.append(f"Word: {clean}\nContext: {extract_clean_text(line)}")
                break
        if len(results) >= max_results:
            break
    frequency = count_tag_frequency(tag)
    return f"Frequency: {frequency}\n{format_results(results)}"

In [60]:
def count_tag_combination_frequency(tags):
    counter = 0
    for line in corpus:
        if all(tag in line for tag in tags):
            counter += 1
    return counter

In [61]:
def search_by_tag_combination(tags, max_results=5):
    results = []
    for line in corpus:
        if all(re.search(f"sem=['\"][^'\"]*{tag}[^'\"]*['\"]", line) for tag in tags):
            words_with_tags = []
            words = line.split()
            for word in words:
                for tag in tags:
                    if re.search(f"sem=['\"][^'\"]*{tag}[^'\"]*['\"]", word):
                        clean_word = word.split('>')[1].split('<')[0]
                        words_with_tags.append(f"Word: {clean_word}")
            if words_with_tags:
                context = extract_clean_text(line)
                for word in words_with_tags:
                    results.append(f"{word}\nContext: {context}")
            if len(results) >= max_results:
                break
    frequency = count_tag_combination_frequency(tags)
    return f"Frequency: {frequency}\n{format_results(results)}"

In [62]:
def count_gr_frequency(tag):
    counter = 0
    # считаем контексты, где встречается граммем
    for line in corpus:
        if re.search(fr"<ana [^>]*gr=['\"][^'\"]*{tag}[^'\"]*['\"]", line):
            counter += 1
    return counter

In [84]:
def search_by_gramm_tag(tag, max_results=5):
    results = []
    pattern = re.compile(fr"<w><ana [^>]*gr=['\"][^'\"]*{tag}[^'\"]*['\"][^>]*>([^<]+)</w>")
    for line in corpus:
        m = pattern.search(line)
        if m:
            word = m.group(1)
            context = extract_clean_text(line)
            results.append(f"Word: {word}\n Context: {context}")
            if len(results) >= max_results:
                break
    frequency = count_gr_frequency(tag)
    return f"Frequency: {frequency}\n {format_results(results)}"

In [64]:
def count_gr_combination_frequency(tags):
    counter = 0
    # считаем контексты, где встречаются все граммемы
    for line in corpus:
        if all(re.search(fr"<ana [^>]*gr=['\"][^'\"]*{tag}[^'\"]*['\"]", line) for tag in tags):
            counter += 1
    return counter

In [65]:
def search_by_gramm_combination(tags, max_results=5):
    results = []
    # шаблоны для каждого тега
    patterns = [re.compile(fr"<w><ana [^>]*gr=['\"][^'\"]*{tag}[^'\"]*['\"][^>]*>([^<]+)</w>") for tag in tags]
    for line in corpus:
        if all(p.search(line) for p in patterns):
            context = extract_clean_text(line)
            for p in patterns:
                m = p.search(line)
                word = m.group(1)
                results.append(f"Word: {word}\n Context: {context}")
                if len(results) >= max_results:
                    return f"Frequency: {count_gr_combination_frequency(tags)}\n {format_results(results)}"
    return f"Frequency: {count_gr_combination_frequency(tags)}\n {format_results(results)}"

In [66]:
if __name__ == "__main__":
    token_name = 'рижском'
    token_results = search_by_token(token_name, max_results=3)
    print(f"Contexts for token '{token_name}':\n{token_results}")

Contexts for token 'рижском':
Frequency: 1
  В  рижском  спектакле  сценография  Ф  Вогербауэра  была  лаконична  но  всё  же  воссоздавала  гумилевскую  атмосферу  старой  заводи  где  есть дом с  голубыми  ставнями  с  креслами  давними  а  мелодии  латышского  маэстро  Р  Паулса  и  итальянского  А  Аннеккино  были  вполне  созвучны  настроению  как  писали  в  старину  Треплева  шопеновских  вальсов 


In [41]:
if __name__ == "__main__":
    lemma_name = 'рижский'
    lemma_results = search_by_lemma(lemma_name, max_results=3)
    print(f"Contexts for lemma '{lemma_name}':\n{lemma_results}")

Contexts for lemma 'рижский':
Frequency: 2
  В  рижском  спектакле  сценография  Ф  Вогербауэра  была  лаконична  но  всё  же  воссоздавала  гумилевскую  атмосферу  старой  заводи  где  есть дом с  голубыми  ставнями  с  креслами  давними  а  мелодии  латышского  маэстро  Р  Паулса  и  итальянского  А  Аннеккино  были  вполне  созвучны  настроению  как  писали  в  старину  Треплева  шопеновских  вальсов 
  В  июле  Кирилл   уехал  со  студенческим  отрядом  в  Новгород  а  мы  с  Ритой  в  конце  июля  взяли  путёвки  на  Рижское  взморье  поехали  немного  раньше  пожили  в  гостинице  а  с  августа      поселились  в доме отдыха 


In [67]:
if __name__ == "__main__":
    semantic_tag_name = 'r:qual'
    semantic_results = search_by_semantic_tag(semantic_tag_name, max_results=3)
    print(f"Contexts for semantic tag '{semantic_tag_name}':\n{semantic_results}")

Contexts for semantic tag 'r:qual':
Frequency: 1396
Word: побогаче
Context:   Должники  возвращали  долг  тем  что  приводили  в дом к  Бертеньеву  новых  клиентов  из  тех  что  были  уже  покрупнее  побогаче
Word: сложным
Context:   Прикрепив  на  разной  высоте  полусферические  контейнеры  с  ампельными  растениями  непосредственно  к  стене дома или  ограде  можно  не  прибегая  к  сложным  конструкциям  декорировать  достаточно  большие  плоскости 
Word: собственный
Context:   Ваши  родственники  или  знакомые  затеяли  строить  собственный дом


In [43]:
if __name__ == "__main__":
    tag_combination_name = ["r:rel", "t:constr"]
    combination_results = search_by_tag_combination(tag_combination_name, max_results=3)
    print(f"Contexts for tag combination '{', '.join(tag_combination_name)}':\n{combination_results}")

Contexts for tag combination 'r:rel, t:constr':
Frequency: 1861
Word: лиственных
Context:   Например  декорирование  фасада дома  беседки  или  даже  ограды  шпалерами  из  хвойных  и  лиственных  растений 
Word: древнем
Context:   Во  время  работы  жюри  работы  были  выставлены  в Доме Европы  затем  перекочевали  на  недельку  во  Дворец  творчества  детей  и  юношества  на  Воробьевых  горах  а  сегодня  выставка  этих  работ  наконец  то  открывается  в  помещении  самого  музея  в  Щетининском  переулке  в  древнем  Замоскворечье 
Word: декоративными
Context:   От  средневекового  города  осталось  лишь  несколько  кварталов  с  декоративными  бревенчатыми домами  собор  Святого  Петра  Дворец  правосудия 


In [85]:
if __name__ == "__main__":
    gramm_tag = 'NUM'
    print(f"Contexts for grammatical tag '{gramm_tag}':\n{search_by_gramm_tag(gramm_tag,3)}")

Contexts for grammatical tag 'NUM':
Frequency: 739
 Word: три
 Context:   Входим  в дом  открыв  дверь  попадаем  в  кухню  печь  налево  окно  рядом  небольшой  стол  керосиновая  лампа  и  три  деревянных  стула 
Word: несколько
 Context:   В  довоенные  годы  Мария  Ивановна  Тарковская  с  детьми  по  несколько  месяцев  жила  в доме номер  8  по  улице  Энгельса 
Word: первой
 Context:   В  первой  половине  апреля  на  сцене  Балтийского дома  состоялся  6-й  международный  фестиваль  русских  театров  СНГ  и  Балтии  Встречи  в  России  собравший  в  Петербурге  спектакли  из  Белоруссии  и  Украины  Эстонии  Латвии  и  Армении 


In [83]:
if __name__ == "__main__":
    gramm_comb = ['S','anim=pl']
    print(f"Contexts for grammatical combination '{', '.join(gramm_comb)}':\n{search_by_gramm_combination(gramm_comb,3)}")

Contexts for grammatical combination 'S, anim=pl':
Frequency: 796
 Word: годы
 Context:   В  довоенные  годы  Мария  Ивановна  Тарковская  с  детьми  по  несколько  месяцев  жила  в доме номер  8  по  улице  Энгельса 
Word: детьми
 Context:   В  довоенные  годы  Мария  Ивановна  Тарковская  с  детьми  по  несколько  месяцев  жила  в доме номер  8  по  улице  Энгельса 
Word: обитатели
 Context:   Даже  местечковые  обитатели  которые  выглядывают  из-за  углов  среди  условных  движущихся  стен  без домов  пластичны  как  в  балете  Якобсона 
